# Lab 4 — Tool-Use Patterns (§19.2.4)

| Pattern | Description | Example in this lab |
|---|---|---|
| Single-turn | one tool call, 2 LLM calls | calculator |
| Multi-tool | several tools at once in one response | morning brief |
| Sequential | one tool's output is the next tool's input | search → fetch |
| Nested | a tool call runs another agent | math sub-agent |
| Fallback | main tool fails → try a backup | primary → backup → cache |

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.2.4 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **4 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`lab4-tool-use.en.ipynb`** (the same code as in the video).

## 0. Setup

You need Python 3.10+ (or Colab) and an **OpenRouter API key** — setup instructions are in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- from Lab 1: client + tools + react_agent (+ LLM call counter and tool timing) ----
import os, json, time
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"enabled": False}}  # this lab doesn't use the Thought
CALLS = {"n": 0}

def ask_llm(messages, tools=None, system=None):
    if system:
        messages = [{"role": "system", "content": system}] + messages
    kwargs = dict(model=MODEL, messages=messages, extra_body=EXTRA)
    if tools:
        kwargs["tools"] = tools
    CALLS["n"] += 1
    return client.chat.completions.create(**kwargs).choices[0].message

TOOLS = {}

def tool(description, properties):
    def wrap(fn):
        TOOLS[fn.__name__] = {"fn": fn, "schema": {"type": "function", "function": {
            "name": fn.__name__, "description": description,
            "parameters": {"type": "object", "properties": properties,
                           "required": list(properties), "additionalProperties": False}}}}
        return fn
    return wrap

def schemas(names):
    return [TOOLS[n]["schema"] for n in names]

def run_tool(call):
    try:
        out = TOOLS[call.function.name]["fn"](**json.loads(call.function.arguments))
    except Exception as e:
        out = f"Error: {e}"
    return {"role": "tool", "tool_call_id": call.id, "content": str(out)}

def react_agent(task, tool_names, system=None, max_iterations=8, indent=""):
    messages = [{"role": "user", "content": task}]
    for i in range(1, max_iterations + 1):
        msg = ask_llm(messages, tools=schemas(tool_names), system=system)
        calls = msg.tool_calls or []
        if not calls:
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [c.model_dump() for c in calls]})
        for c in calls:
            print(f"{indent}🔧 [{i}] {c.function.name}({c.function.arguments})")
        t0 = time.time()
        with ThreadPoolExecutor() as pool:
            results = list(pool.map(run_tool, calls))
        if len(calls) > 1:
            print(f"{indent}⏱️ {len(calls)} tools ran in parallel: {time.time() - t0:.1f}s")
        for r in results:
            print(f"{indent}👀 [{i}] {r['content'][:120]}")
        messages.extend(results)
    return "⚠️ Stopped: reached max_iterations"

## 1. Single-turn — one tool, 2 LLM calls

In [ ]:
# ✏️ TODO 1: Guard eval: allow only digits, + - * / ( ) . and spaces
@tool("Evaluate an arithmetic expression, e.g. '(3+4)*2'.", {"expression": {"type": "string"}})
def calculator(expression):
    if not ___:
        return "Error: only numbers and + - * / ( ) are allowed"
    return str(eval(expression))

In [ ]:
before = CALLS["n"]
print(react_agent("What is (17 * 23) + 9? Use the calculator.", ["calculator"]))
print("LLM calls:", CALLS["n"] - before)

## 2. Multi-tool (parallel)

The tools must be **independent of each other** → the harness runs them at the same time → **latency = max(...)** instead of the sum

In [ ]:
def slow(value):
    time.sleep(1.5)
    return value

@tool("Get the latest stock price for a ticker.", {"ticker": {"type": "string"}})
def get_stock_price(ticker):
    return slow(json.dumps({"ticker": ticker, "price_usd": 187.4}))

@tool("Get today's weather for a city.", {"city": {"type": "string"}})
def get_weather(city):
    return slow(json.dumps({"city": city, "forecast": "rain, 29°C"}, ensure_ascii=False))

@tool("Get calendar events for a date (YYYY-MM-DD).", {"date": {"type": "string"}})
def get_calendar(date):
    return slow(json.dumps([{"time": "14:00", "event": "Thesis meeting"}]))

In [ ]:
import datetime
today = datetime.date.today().isoformat()
print(react_agent(f"Today is {today}. Give me a morning brief: AAPL stock price, Bangkok weather, "
                  "and my calendar for today. Fetch all three at once.",
                  ["get_stock_price", "get_weather", "get_calendar"]))

## 3. Sequential (pipeline)

One tool's output is the next tool's input — the model picks the next tool based on the previous result

In [ ]:
DOCS = {  # simulated data
    "https://docs.example/patchtst": "PatchTST splits each series into patches of 16 steps "
                                     "and models each channel independently.",
    "https://docs.example/itransformer": "iTransformer applies attention across variates instead of time steps.",
}

# ✏️ TODO 2: fetch_page: return the content from DOCS for the url; if that url is missing, return an Error message instead of crashing
@tool("Search the docs. Returns a JSON list of {title, url}.", {"query": {"type": "string"}})
def search_docs(query):
    return json.dumps([{"title": u.rsplit("/", 1)[1], "url": u} for u in DOCS])

@tool("Fetch a page by url and return its text.", {"url": {"type": "string"}})
def fetch_page(url):
    return ___

In [ ]:
print(react_agent("How long is each patch in PatchTST? Search the docs, then fetch the right page.",
                  ["search_docs", "fetch_page"]))

## 4. Nested (agent-as-tool)

The parent agent sees the sub-agent as just another tool (a black box), but inside it is another agent with its own prompt and tools

In [ ]:
# ✏️ TODO 3: agent-as-tool: inside, this tool is another react_agent with calculator as its tool
@tool("Delegate a math word problem to a specialist math agent. Returns the worked answer.",
      {"problem": {"type": "string"}})
def ask_math_agent(problem):
    print("      ↳ 🤖 math sub-agent started")
    return ___(problem, ___,
               system="You are a careful math agent. Always use the calculator for arithmetic.",
               indent="      ")

In [ ]:
print(react_agent("A class has 3 sections of 42 students. Each student needs 2 notebooks "
                  "costing 18.5 baht each. How much do all notebooks cost? Delegate the math.",
                  ["ask_math_agent"]))

## 5. Fallback (graceful degradation)

The harness tries tools in priority order — **the model doesn't need to know about the fallback at all** (the *Plan for failure* principle, §19.3)

In [ ]:
def primary_search(q): raise TimeoutError("primary API timed out")
def backup_search(q):  raise RuntimeError("429 rate limited")
def cached_search(q):  return json.dumps([{"title": "PatchTST", "source": "cache (2 days old)"}])

# ✏️ TODO 4: Try the search sources one by one in order; return as soon as one succeeds; if all fail, return a polite message telling the model
@tool("Search the web for a topic.", {"query": {"type": "string"}})
def web_search(query):
    for name, fn in [("primary", primary_search), ("backup", backup_search), ("cache", cached_search)]:
        try:
            result = ___
            print(f"      ✅ {name} succeeded")
            return result
        except Exception as e:
            print(f"      ❌ {name} failed: {e}")
    return ___

In [ ]:
print(react_agent("Search the web for a patch-based time-series transformer and name one.", ["web_search"]))

## Summary

| Pattern | What we saw in the lab |
|---|---|
| Single-turn | LLM calls = 2 |
| Multi-tool | 3 tools took ~1.5s, not 4.5s |
| Sequential | search_docs → fetch_page |
| Nested | a sub-agent with its own prompt + tools |
| Fallback | primary ❌ → backup ❌ → cache ✅ |

**Try next:** create a tool that needs another tool's result, then ask the model to "fetch all at once" — what happens?